# 07 - Reservation Policy Demand Sensitivity

This notebook focuses on demand regimes for the strict reservation policy. FCFS is included as a light reference because it was already explored in earlier notebooks.

Policy definitions:

- **Pooled FCFS:** no reserved slots.
- **Strict C1 reservation:** `Q` slots per day are reserved for Class 1. Class 1 searches appointment days chronologically, trying reserved capacity before general capacity within the same day. Class 2 can only use general capacity. Unused reserved slots stay empty.


## Imports And Repo Setup

This uses the same simple repo-root detection pattern as notebook 05. No YAML files are read or written.

In [ ]:
from __future__ import annotations

from copy import deepcopy
from dataclasses import replace
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable=None, **kwargs):
        return iterable


def find_repo_dir(start: Path) -> Path:
    current = Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "simulation" / "engine.py").exists() and (
            candidate / "analysis" / "metrics.py"
        ).exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root from the current notebook location.")


REPO_DIR = find_repo_dir(Path.cwd())
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from analysis.metrics import aggregate_result_row, class_result_rows
from simulation.engine import ClinicAppointmentSimulation
from simulation.model import PatientClassParams, SimulationConfig, ThresholdRule

plt.style.use("default")

## Base Scenario And Policy Definitions

The base scenario fixes capacity and behavior. Each experiment below changes only the two arrival rates unless noted otherwise.

In [ ]:
BASE_SCENARIO = {
    "slots_per_day": 32,
    "reserved_slots_per_day": 10,
    "reserved_class_id": 1,
    "horizon_days": 14,
    "burn_in_days": 30,
    "measure_days": 365,
    "cooldown_days": 14,
    "seeds": list(range(5101, 5131)),
    "classes": {
        1: {
            "lambda_per_day": 25.0,
            "cancel_prob": 0.10,
            "value": 1.0,
            "balk_prob": {"threshold": 9, "low": 0.00, "high": 0.50},
            "no_show_prob": {"threshold": 6, "low": 0.00, "high": 0.30},
        },
        2: {
            "lambda_per_day": 25.0,
            "cancel_prob": 0.10,
            "value": 1.0,
            "balk_prob": {"threshold": 9, "low": 0.00, "high": 0.50},
            "no_show_prob": {"threshold": 6, "low": 0.00, "high": 0.30},
        },
    },
}

POLICIES = ["Pooled FCFS", "Strict C1 reservation"]
RESERVATION_POLICIES = ["Strict C1 reservation"]
POLICY_COLORS = {
    "Pooled FCFS": "0.45",
    "Strict C1 reservation": "tab:blue",
}
LOST_COMPONENTS = [
    "balked_rate",
    "canceled_rate",
    "no_show_rate",
    "no_offer_rate",
    "unresolved_booked_rate",
]

scenario_table = pd.DataFrame(
    {
        "value": pd.Series(
            {
                key: value
                for key, value in BASE_SCENARIO.items()
                if key not in {"classes", "seeds"}
            },
            dtype="object",
        )
    }
)
scenario_table.loc["num_seeds", "value"] = len(BASE_SCENARIO["seeds"])
scenario_table.loc["seed_range", "value"] = f"{BASE_SCENARIO['seeds'][0]}-{BASE_SCENARIO['seeds'][-1]}"

class_table = pd.DataFrame(
    [
        {
            "class_id": class_id,
            "lambda_per_day": params["lambda_per_day"],
            "cancel_prob": params["cancel_prob"],
            "balk_threshold": params["balk_prob"]["threshold"],
            "balk_low": params["balk_prob"]["low"],
            "balk_high": params["balk_prob"]["high"],
            "no_show_threshold": params["no_show_prob"]["threshold"],
            "no_show_low": params["no_show_prob"]["low"],
            "no_show_high": params["no_show_prob"]["high"],
        }
        for class_id, params in BASE_SCENARIO["classes"].items()
    ]
)

display(scenario_table)
display(class_table)

## Small Helpers

Only six notebook helpers are defined here. The simulation and row-level metric logic remain in the existing project modules.

In [ ]:
def build_config(scenario: dict, policy: str, seed: int | None) -> SimulationConfig:
    classes = {}
    for class_id, params in scenario["classes"].items():
        classes[class_id] = PatientClassParams(
            class_id=class_id,
            lambda_per_day=float(params["lambda_per_day"]),
            balk_prob=ThresholdRule(**params["balk_prob"]),
            cancel_prob=float(params["cancel_prob"]),
            no_show_prob=ThresholdRule(**params["no_show_prob"]),
            value=float(params.get("value", 1.0)),
        )

    reserved = policy != "Pooled FCFS"
    reserved_slots = int(scenario["reserved_slots_per_day"]) if reserved else 0

    return SimulationConfig(
        slots_per_day=int(scenario["slots_per_day"]),
        horizon_days=int(scenario["horizon_days"]),
        burn_in_days=int(scenario["burn_in_days"]),
        measure_days=int(scenario["measure_days"]),
        cooldown_days=int(scenario["cooldown_days"]),
        classes=classes,
        seed=seed,
        reserved_class_id=scenario["reserved_class_id"] if reserved_slots > 0 else None,
        reserved_slots_per_day=reserved_slots,
    )


def make_scenario(lambda_1: float, lambda_2: float, base_scenario: dict = BASE_SCENARIO) -> dict:
    scenario = deepcopy(base_scenario)
    scenario["classes"][1]["lambda_per_day"] = float(lambda_1)
    scenario["classes"][2]["lambda_per_day"] = float(lambda_2)
    return scenario


def add_rates(aggregate_df: pd.DataFrame, class_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    aggregate_df = aggregate_df.copy()
    class_df = class_df.copy()

    arrivals = aggregate_df["total_arrivals"]
    aggregate_df["served_rate"] = aggregate_df["total_served"].div(arrivals).where(arrivals != 0, 0.0)
    aggregate_df["balked_rate"] = aggregate_df["total_balked"].div(arrivals).where(arrivals != 0, 0.0)
    aggregate_df["no_offer_rate"] = aggregate_df["total_no_offer"].div(arrivals).where(arrivals != 0, 0.0)
    aggregate_df["canceled_rate"] = aggregate_df["total_canceled"].div(arrivals).where(arrivals != 0, 0.0)
    aggregate_df["no_show_rate"] = aggregate_df["total_no_show"].div(arrivals).where(arrivals != 0, 0.0)
    aggregate_df["unresolved_booked_rate"] = aggregate_df["total_unresolved_booked"].div(arrivals).where(arrivals != 0, 0.0)
    aggregate_df["lost_components_sum"] = aggregate_df[LOST_COMPONENTS].sum(axis=1)
    aggregate_df["lost_rate"] = 1.0 - aggregate_df["served_rate"]

    class_df["unresolved_booked"] = class_df["booked"] - class_df["canceled"] - class_df["no_show"] - class_df["served"]
    class_arrivals = class_df["arrivals"]
    class_df["served_rate"] = class_df["percent_serviced"]
    class_df["balked_rate"] = class_df["balked"].div(class_arrivals).where(class_arrivals != 0, 0.0)
    class_df["no_offer_rate"] = class_df["no_offer"].div(class_arrivals).where(class_arrivals != 0, 0.0)
    class_df["canceled_rate"] = class_df["canceled"].div(class_arrivals).where(class_arrivals != 0, 0.0)
    class_df["no_show_rate"] = class_df["no_show"].div(class_arrivals).where(class_arrivals != 0, 0.0)
    class_df["unresolved_booked_rate"] = class_df["unresolved_booked"].div(class_arrivals).where(class_arrivals != 0, 0.0)
    class_df["lost_components_sum"] = class_df[LOST_COMPONENTS].sum(axis=1)
    class_df["lost_rate"] = 1.0 - class_df["served_rate"]

    return aggregate_df, class_df


def validate_accounting(aggregate_df: pd.DataFrame, class_df: pd.DataFrame, tol: float = 1e-9) -> None:
    aggregate_partition = (
        aggregate_df["total_served"]
        + aggregate_df["total_balked"]
        + aggregate_df["total_no_offer"]
        + aggregate_df["total_canceled"]
        + aggregate_df["total_no_show"]
        + aggregate_df["total_unresolved_booked"]
    )
    if (aggregate_partition - aggregate_df["total_arrivals"]).abs().max() > tol:
        raise AssertionError("Aggregate outcomes do not partition arrivals.")
    if (aggregate_df["total_unresolved_booked"] < -tol).any():
        raise AssertionError("Aggregate unresolved_booked is negative.")
    if (aggregate_df["lost_rate"] - (1.0 - aggregate_df["served_rate"])).abs().max() > tol:
        raise AssertionError("Aggregate lost_rate is not 1 - served_rate.")
    if (aggregate_df["lost_components_sum"] - aggregate_df["lost_rate"]).abs().max() > tol:
        raise AssertionError("Aggregate lost components do not sum to lost_rate.")

    class_partition = (
        class_df["served"]
        + class_df["balked"]
        + class_df["no_offer"]
        + class_df["canceled"]
        + class_df["no_show"]
        + class_df["unresolved_booked"]
    )
    if (class_partition - class_df["arrivals"]).abs().max() > tol:
        raise AssertionError("Class outcomes do not partition arrivals.")
    if (class_df["unresolved_booked"] < -tol).any():
        raise AssertionError("Class unresolved_booked is negative.")
    if (class_df["lost_components_sum"] - class_df["lost_rate"]).abs().max() > tol:
        raise AssertionError("Class lost components do not sum to lost_rate.")


def run_experiment(experiment_rows: list[dict], include_fcfs: bool = True) -> tuple[pd.DataFrame, pd.DataFrame]:
    policies = POLICIES if include_fcfs else RESERVATION_POLICIES
    aggregate_rows = []
    class_rows = []

    for experiment in tqdm(experiment_rows, desc="experiment settings"):
        scenario = make_scenario(experiment["lambda_1"], experiment["lambda_2"], experiment.get("base_scenario", BASE_SCENARIO))
        metadata = {key: value for key, value in experiment.items() if key != "base_scenario"}

        for policy in policies:
            for seed in scenario["seeds"]:
                config = build_config(scenario, policy, seed=int(seed))
                result = ClinicAppointmentSimulation(config).run()
                fixed_values = {"policy": policy, "seed": int(seed), **metadata}
                aggregate_rows.append(aggregate_result_row(result, fixed_values))
                class_rows.extend(class_result_rows(result, fixed_values))

    aggregate_df = pd.DataFrame(aggregate_rows)
    class_df = pd.DataFrame(class_rows)
    aggregate_df, class_df = add_rates(aggregate_df, class_df)
    validate_accounting(aggregate_df, class_df)
    return aggregate_df, class_df

## Baseline Sanity Check

Balanced demand: `lambda_1 = 25`, `lambda_2 = 25`.

In [ ]:
baseline_rows = [{"lambda_1": 25, "lambda_2": 25, "case": "baseline"}]
baseline_aggregate_df, baseline_class_df = run_experiment(baseline_rows, include_fcfs=True)

baseline_aggregate_summary = (
    baseline_aggregate_df.groupby("policy")[[
        "served_rate",
        "average_utilization",
        "mean_offered_booking_delay",
        "lost_rate",
        "balked_rate",
        "canceled_rate",
        "no_show_rate",
        "no_offer_rate",
        "unresolved_booked_rate",
    ]]
    .mean()
    .reindex(POLICIES)
    .round(4)
)

baseline_class_summary = (
    baseline_class_df.groupby(["policy", "class_id"])[[
        "served_rate",
        "mean_offered_booking_delay",
        "lost_rate",
        "balked_rate",
        "canceled_rate",
        "no_show_rate",
        "no_offer_rate",
        "unresolved_booked_rate",
    ]]
    .mean()
    .round(4)
)

display(baseline_aggregate_summary)
display(baseline_class_summary)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

served_table = (
    baseline_class_df.groupby(["policy", "class_id"])["served_rate"]
    .mean()
    .unstack("class_id")
    .reindex(POLICIES)
)
served_table.plot(kind="bar", ax=axes[0])
axes[0].set_title("Served Rate By Class")
axes[0].set_xlabel("policy")
axes[0].set_ylabel("served rate")
axes[0].grid(axis="y", alpha=0.25)
axes[0].tick_params(axis="x", rotation=25)
axes[0].legend(title="class", frameon=False)

wait_table = (
    baseline_class_df.groupby(["policy", "class_id"])["mean_offered_booking_delay"]
    .mean()
    .unstack("class_id")
    .reindex(POLICIES)
)
wait_table.plot(kind="bar", ax=axes[1])
axes[1].set_title("Mean Offered Delay By Class")
axes[1].set_xlabel("policy")
axes[1].set_ylabel("days")
axes[1].grid(axis="y", alpha=0.25)
axes[1].tick_params(axis="x", rotation=25)
axes[1].legend(title="class", frameon=False)

utilization = baseline_aggregate_df.groupby("policy")["average_utilization"].mean().reindex(POLICIES)
utilization.plot(kind="bar", ax=axes[2], color=[POLICY_COLORS[p] for p in utilization.index])
axes[2].set_title("Average Utilization")
axes[2].set_xlabel("policy")
axes[2].set_ylabel("served-slot utilization")
axes[2].grid(axis="y", alpha=0.25)
axes[2].tick_params(axis="x", rotation=25)

fig.tight_layout()

## Symmetric Total-Demand Regime Sweep

Purpose: observe the transition from under-capacity to overload. Demand is split evenly between classes.

In [ ]:
lambda_total_values = [10, 20, 32, 40, 60, 80, 120, 160]

total_demand_rows = []
for lambda_total in lambda_total_values:
    if lambda_total < BASE_SCENARIO["slots_per_day"]:
        regime = "under_capacity"
    elif abs(lambda_total - BASE_SCENARIO["slots_per_day"]) <= 0.10 * BASE_SCENARIO["slots_per_day"]:
        regime = "near_capacity"
    elif lambda_total < 80:
        regime = "overloaded"
    else:
        regime = "extreme_overload"

    total_demand_rows.append(
        {
            "lambda_total": lambda_total,
            "lambda_1": lambda_total / 2,
            "lambda_2": lambda_total / 2,
            "regime": regime,
        }
    )

total_aggregate_df, total_class_df = run_experiment(total_demand_rows, include_fcfs=True)

display(
    total_aggregate_df.groupby(["lambda_total", "regime", "policy"])[[
        "served_rate",
        "average_utilization",
        "mean_offered_booking_delay",
        "lost_rate",
    ]]
    .mean()
    .round(4)
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

for policy in POLICIES:
    policy_df = total_aggregate_df[total_aggregate_df["policy"] == policy]
    line = policy_df.groupby("lambda_total")["served_rate"].mean().sort_index()
    axes[0].plot(
        line.index,
        line.values,
        marker="o",
        linestyle="--" if policy == "Pooled FCFS" else "-",
        color=POLICY_COLORS[policy],
        label=policy,
    )
axes[0].set_title("Overall Served Rate")
axes[0].set_xlabel("total expected daily arrivals")
axes[0].set_ylabel("served rate")
axes[0].grid(axis="y", alpha=0.25)
axes[0].legend(frameon=False, fontsize=9)

for policy in POLICIES:
    policy_df = total_aggregate_df[total_aggregate_df["policy"] == policy]
    line = policy_df.groupby("lambda_total")["average_utilization"].mean().sort_index()
    axes[1].plot(
        line.index,
        line.values,
        marker="o",
        linestyle="--" if policy == "Pooled FCFS" else "-",
        color=POLICY_COLORS[policy],
        label=policy,
    )
axes[1].set_title("Average Utilization")
axes[1].set_xlabel("total expected daily arrivals")
axes[1].set_ylabel("served-slot utilization")
axes[1].grid(axis="y", alpha=0.25)
axes[1].legend(frameon=False, fontsize=9)

for policy in POLICIES:
    policy_df = total_aggregate_df[total_aggregate_df["policy"] == policy]
    line = policy_df.groupby("lambda_total")["mean_offered_booking_delay"].mean().sort_index()
    axes[2].plot(
        line.index,
        line.values,
        marker="o",
        linestyle="--" if policy == "Pooled FCFS" else "-",
        color=POLICY_COLORS[policy],
        label=policy,
    )
axes[2].set_title("Mean Offered Delay")
axes[2].set_xlabel("total expected daily arrivals")
axes[2].set_ylabel("days")
axes[2].grid(axis="y", alpha=0.25)
axes[2].legend(frameon=False, fontsize=9)

fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
reservation_class_df = total_class_df[total_class_df["policy"].isin(RESERVATION_POLICIES)]

for (policy, class_id), group in reservation_class_df.groupby(["policy", "class_id"]):
    line = group.groupby("lambda_total")["served_rate"].mean().sort_index()
    axes[0].plot(
        line.index,
        line.values,
        marker="o" if class_id == 1 else "s",
        color=POLICY_COLORS[policy],
        label=f"{policy} C{class_id}",
    )
axes[0].set_title("Class-Specific Served Rate")
axes[0].set_xlabel("total expected daily arrivals")
axes[0].set_ylabel("served rate")
axes[0].grid(axis="y", alpha=0.25)
axes[0].legend(frameon=False, fontsize=8)

for (policy, class_id), group in reservation_class_df.groupby(["policy", "class_id"]):
    line = group.groupby("lambda_total")["mean_offered_booking_delay"].mean().sort_index()
    axes[1].plot(
        line.index,
        line.values,
        marker="o" if class_id == 1 else "s",
        color=POLICY_COLORS[policy],
        label=f"{policy} C{class_id}",
    )
axes[1].set_title("Class-Specific Mean Offered Delay")
axes[1].set_xlabel("total expected daily arrivals")
axes[1].set_ylabel("days")
axes[1].grid(axis="y", alpha=0.25)
axes[1].legend(frameon=False, fontsize=8)

fig.tight_layout()

### Interpretation Placeholders

- Under-capacity regime:
- Near-capacity regime:
- Overloaded regime:
- Extreme-overload regime:

## Class 1 Demand Concentration Sweep

Hold `lambda_2 = 25` and vary Class 1 demand.

In [ ]:
lambda_1_values = [2, 5, 10, 20, 32, 50, 80, 120]
fixed_lambda_2 = 25
c1_rows = []

for lambda_1 in lambda_1_values:
    if lambda_1 >= 80:
        regime = "extreme_priority_demand"
    elif lambda_1 < BASE_SCENARIO["reserved_slots_per_day"]:
        regime = "below_reserved_capacity"
    elif abs(lambda_1 - BASE_SCENARIO["reserved_slots_per_day"]) <= 0.25 * BASE_SCENARIO["reserved_slots_per_day"]:
        regime = "around_reserved_capacity"
    else:
        regime = "above_reserved_capacity"

    c1_rows.append(
        {
            "lambda_1": lambda_1,
            "lambda_2": fixed_lambda_2,
            "lambda_total": lambda_1 + fixed_lambda_2,
            "regime": regime,
        }
    )

c1_aggregate_df, c1_class_df = run_experiment(c1_rows, include_fcfs=False)

display(
    c1_aggregate_df.groupby(["lambda_1", "regime", "policy"])[[
        "served_rate",
        "average_utilization",
        "mean_offered_booking_delay",
        "lost_rate",
    ]]
    .mean()
    .round(4)
)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
axes = axes.ravel()

for (policy, class_id), group in c1_class_df.groupby(["policy", "class_id"]):
    line = group.groupby("lambda_1")["served_rate"].mean().sort_index()
    axes[0].plot(line.index, line.values, marker="o" if class_id == 1 else "s", color=POLICY_COLORS[policy], label=f"{policy} C{class_id}")
axes[0].set_title("Class Served Rate")
axes[0].set_xlabel("Class 1 lambda")
axes[0].set_ylabel("served rate")
axes[0].grid(axis="y", alpha=0.25)
axes[0].legend(frameon=False, fontsize=8)

for (policy, class_id), group in c1_class_df.groupby(["policy", "class_id"]):
    line = group.groupby("lambda_1")["mean_offered_booking_delay"].mean().sort_index()
    axes[1].plot(line.index, line.values, marker="o" if class_id == 1 else "s", color=POLICY_COLORS[policy], label=f"{policy} C{class_id}")
axes[1].set_title("Class Mean Offered Delay")
axes[1].set_xlabel("Class 1 lambda")
axes[1].set_ylabel("days")
axes[1].grid(axis="y", alpha=0.25)
axes[1].legend(frameon=False, fontsize=8)

for policy in RESERVATION_POLICIES:
    policy_df = c1_aggregate_df[c1_aggregate_df["policy"] == policy]
    line = policy_df.groupby("lambda_1")["average_utilization"].mean().sort_index()
    axes[2].plot(line.index, line.values, marker="o", color=POLICY_COLORS[policy], label=policy)
axes[2].set_title("Average Utilization")
axes[2].set_xlabel("Class 1 lambda")
axes[2].set_ylabel("served-slot utilization")
axes[2].grid(axis="y", alpha=0.25)
axes[2].legend(frameon=False, fontsize=9)

tradeoff = (
    c1_class_df.groupby(["policy", "lambda_1", "class_id"])["served_rate"]
    .mean()
    .unstack("class_id")
    .rename(columns={1: "class_1_served_rate", 2: "class_2_served_rate"})
    .reset_index()
)
for policy in RESERVATION_POLICIES:
    policy_df = tradeoff[tradeoff["policy"] == policy].sort_values("lambda_1")
    axes[3].plot(policy_df["class_2_served_rate"], policy_df["class_1_served_rate"], marker="o", color=POLICY_COLORS[policy], label=policy)
    for _, row in policy_df.iterrows():
        axes[3].annotate(str(int(row["lambda_1"])), (row["class_2_served_rate"], row["class_1_served_rate"]), textcoords="offset points", xytext=(4, 4), fontsize=8)
axes[3].set_title("Served-Rate Tradeoff")
axes[3].set_xlabel("Class 2 served rate")
axes[3].set_ylabel("Class 1 served rate")
axes[3].grid(axis="y", alpha=0.25)
axes[3].legend(frameon=False, fontsize=9)

fig.tight_layout()

## Class 2 Demand Concentration Sweep

Hold `lambda_1 = 25` and vary Class 2 demand.

In [ ]:
lambda_2_values = [2, 5, 10, 20, 32, 50, 80, 120]
fixed_lambda_1 = 25
c2_rows = [
    {
        "lambda_1": fixed_lambda_1,
        "lambda_2": lambda_2,
        "lambda_total": fixed_lambda_1 + lambda_2,
    }
    for lambda_2 in lambda_2_values
]

c2_aggregate_df, c2_class_df = run_experiment(c2_rows, include_fcfs=False)

display(
    c2_aggregate_df.groupby(["lambda_2", "policy"])[[
        "served_rate",
        "average_utilization",
        "mean_offered_booking_delay",
        "lost_rate",
    ]]
    .mean()
    .round(4)
)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
axes = axes.ravel()

for (policy, class_id), group in c2_class_df.groupby(["policy", "class_id"]):
    line = group.groupby("lambda_2")["served_rate"].mean().sort_index()
    axes[0].plot(line.index, line.values, marker="o" if class_id == 1 else "s", color=POLICY_COLORS[policy], label=f"{policy} C{class_id}")
axes[0].set_title("Class Served Rate")
axes[0].set_xlabel("Class 2 lambda")
axes[0].set_ylabel("served rate")
axes[0].grid(axis="y", alpha=0.25)
axes[0].legend(frameon=False, fontsize=8)

for (policy, class_id), group in c2_class_df.groupby(["policy", "class_id"]):
    line = group.groupby("lambda_2")["mean_offered_booking_delay"].mean().sort_index()
    axes[1].plot(line.index, line.values, marker="o" if class_id == 1 else "s", color=POLICY_COLORS[policy], label=f"{policy} C{class_id}")
axes[1].set_title("Class Mean Offered Delay")
axes[1].set_xlabel("Class 2 lambda")
axes[1].set_ylabel("days")
axes[1].grid(axis="y", alpha=0.25)
axes[1].legend(frameon=False, fontsize=8)

for policy in RESERVATION_POLICIES:
    policy_df = c2_aggregate_df[c2_aggregate_df["policy"] == policy]
    line = policy_df.groupby("lambda_2")["average_utilization"].mean().sort_index()
    axes[2].plot(line.index, line.values, marker="o", color=POLICY_COLORS[policy], label=policy)
axes[2].set_title("Average Utilization")
axes[2].set_xlabel("Class 2 lambda")
axes[2].set_ylabel("served-slot utilization")
axes[2].grid(axis="y", alpha=0.25)
axes[2].legend(frameon=False, fontsize=9)

tradeoff = (
    c2_class_df.groupby(["policy", "lambda_2", "class_id"])["served_rate"]
    .mean()
    .unstack("class_id")
    .rename(columns={1: "class_1_served_rate", 2: "class_2_served_rate"})
    .reset_index()
)
for policy in RESERVATION_POLICIES:
    policy_df = tradeoff[tradeoff["policy"] == policy].sort_values("lambda_2")
    axes[3].plot(policy_df["class_2_served_rate"], policy_df["class_1_served_rate"], marker="o", color=POLICY_COLORS[policy], label=policy)
    for _, row in policy_df.iterrows():
        axes[3].annotate(str(int(row["lambda_2"])), (row["class_2_served_rate"], row["class_1_served_rate"]), textcoords="offset points", xytext=(4, 4), fontsize=8)
axes[3].set_title("Served-Rate Tradeoff")
axes[3].set_xlabel("Class 2 served rate")
axes[3].set_ylabel("Class 1 served rate")
axes[3].grid(axis="y", alpha=0.25)
axes[3].legend(frameon=False, fontsize=9)

fig.tight_layout()

## Accounting Checks

`validate_accounting` is called inside `run_experiment`. This final check repeats the assertions across every experiment table in the notebook.

In [ ]:
all_experiments = {
    "baseline": (baseline_aggregate_df, baseline_class_df),
    "symmetric_total_demand": (total_aggregate_df, total_class_df),
    "symmetric_per_class_overload": (per_class_aggregate_df, per_class_class_df),
    "class_1_demand_concentration": (c1_aggregate_df, c1_class_df),
    "class_2_demand_concentration": (c2_aggregate_df, c2_class_df),
}

for name, (aggregate_df, class_df) in all_experiments.items():
    validate_accounting(aggregate_df, class_df)
    print(f"{name}: accounting checks passed")

## Final Notes

- All shares are per measured arrival.
- Mean offered delay is conditional on receiving an offer.
- `cooldown_days` is set to `horizon_days`.
- The same seed does not guarantee perfect common random numbers because policies can consume RNG differently after behavioral divergence.
- This notebook studies demand sensitivity only. `Q` sensitivity and behavioral parameter sensitivity should be separate notebooks.